# Land Cover with Prithvi-EO-2.0
**NB 03** | Ethiopian Highlands | Sentinel-2 | Prithvi 600M

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pathlib

import numpy as np

import pygeovision as pgv

client = pgv.PyGeoVision()
print(client)

from pygeovision.models.foundation.prithvi import PrithviTasks


In [ ]:
BBOX       = (38.6, 8.9, 38.95, 9.15)
DATE_RANGE = ('2024-01-01', '2024-04-30')
PROVIDERS  = ['planetary_computer']

DATA_DIR = pathlib.Path('./results/notebook')
DATA_DIR.mkdir(parents=True, exist_ok=True)

results = client.search(
    bbox            = BBOX,
    date_range      = DATE_RANGE,
    providers       = PROVIDERS,
    cloud_cover_max = 15,
)
print(f"Found {len(results)} scenes")
for r in results[:5]:
    print(f"  {r.provider:<22} {r.datetime[:10]}  cloud={r.cloud_cover:.1f}%  {r.id[:40]}")


In [ ]:
BANDS = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']

downloads = client.download(
    results[:1],
    output_dir   = str(DATA_DIR),
    bands        = BANDS,
    post_process = ['reproject:EPSG:32637', 'cog'],
)
scene_path = downloads[0].path if downloads and downloads[0].success else None
scl_cands  = list(DATA_DIR.rglob('*SCL*.tif')) + list(DATA_DIR.rglob('*scl*.tif'))
scl_path   = str(scl_cands[0]) if scl_cands else None
if scene_path:
    d = downloads[0]
    print(f"Downloaded   : {scene_path}")
    print(f"Size         : {d.bytes_downloaded/1024/1024:.1f} MB")
    print(f"SCL mask     : {scl_path}")
else:
    print("Download failed or scene unavailable — check provider availability")


In [ ]:
if scene_path:
    raw_report = client.validator.validate(
        str(scene_path),
        required_bands = 6,
        value_range    = (0, 65535),
    )
    print("Raw data validation:")
    print(raw_report.summary())
    if not raw_report.passed:
        print("WARNING: Issues found — preprocessing will auto-fix")


In [ ]:
PREPROCESSED = DATA_DIR / 'ethiopia_prithvi_ready.tif'

if scene_path:
    ready = client.prepare_for_ai(
        str(scene_path),
        stack_bands  = BANDS,
        bbox         = BBOX,
        scl_path         = scl_path,
        scl_keep_classes = [4,5,6],
        normalise    = 'scale_factor',
        scale_factor = 10000.0,
        model_type   = 'foundation',
        output_path  = str(PREPROCESSED),
    )
    print("Preprocessing steps :", ready['steps'])
    print("Output shape (C,H,W):", ready['shape'])
    print("Resolution          :", ready['resolution_m'], "m")
    print("Validation          :", "PASSED" if ready['report'] and ready['report'].passed else "FIXED (auto)")
    arr = ready['array']
    print(f"Value range         : {arr.min():.4f} → {arr.max():.4f}")
    print(f"NaN count           : {np.isnan(arr).sum()}")
else:
    print("No scene available — cannot preprocess")
    ready = None


In [ ]:
# Prithvi-EO-2.0 land cover — automatically applies map_bands + normalise_hls
PREDICTION = DATA_DIR / 'land_cover.tif'
CLASS_NAMES = ['Tree cover','Shrubland','Grassland','Cropland',
               'Built-up','Bare/sparse','Snow/ice','Water','Wetland']
if PREPROCESSED.exists():
    tasks = PrithviTasks('prithvi_eo_2_0')
    tasks.land_cover(str(PREPROCESSED), source='sentinel2', output_path=str(PREDICTION))
    print(f"Land cover map saved: {PREDICTION}")


In [ ]:
if PREDICTION.exists():
    pred_report = client.validator.validate(str(PREDICTION))
    print("Prediction validation:", "PASSED" if pred_report.passed else f"FIXED — {pred_report.errors}")
    print(pred_report.summary())


In [ ]:
SIEVED = DATA_DIR / 'lc_sieved.tif'
COG_OUT = DATA_DIR / 'lc_cog.tif'
if PREDICTION.exists():
    client.postprocess.sieve_filter(str(PREDICTION), min_pixels=50, output_path=str(SIEVED))
    stats = client.postprocess.class_statistics(str(SIEVED))
    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    client.postprocess.generate_report(str(SIEVED), str(DATA_DIR/'lc_report.html'))
    print(f"{'Cls':<4} {'Name':<22} {'Area(ha)':>10} {'Cover':>8}")
    print('-'*48)
    for i, name in enumerate(CLASS_NAMES):
        inf = stats.get(i, {})
        print(f"{i:<4} {name:<22} {inf.get('area_ha',0):>10.1f} {inf.get('pct',0):>7.1f}%")


**NB03 — Land Cover** complete.
- Study area: Ethiopian Highlands
- Sensor: Sentinel-2 6-band
- Model: Prithvi-EO-2.0 600M
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG